# 05 - Embedding exploration

This notebook has two parts:

1. **Data setup.** How to download the published full-dataset release from Zenodo, as an alternative to rebuilding it from the raw RDF (see the root `README.md`, "Building the PyG").
2. **Embedding exploration.** PCA, t-SNE and UMAP projections of the Metabolite, Protein and GeneProduct embeddings from `03_embeddings_fulldata.ipynb`, coloured by organism. A diagnostic then checks whether the taxa split places the held-out organisms in a distinct region of embedding space. Part 4 gives the same treatment to the Conversion (reaction) embeddings from `scripts/compute_conversion_embeddings.py`, coloured by EC class instead of organism.

## Part 1 -- Data setup: download the published full-dataset release

The full dataset built in `03_embeddings_fulldata.ipynb` and `02_build_splits.ipynb` is published on Zenodo as its own deposition, separate from the source PlantMetWiki RDF (see `fulldata/zenodo_metadata.json` and `fulldata/LICENSE.md`).

**Concept DOI (always resolves to the latest version):**
[10.5281/zenodo.20847651](https://doi.org/10.5281/zenodo.20847651)

Querying the Zenodo API with the concept record id, rather than a specific version's id, returns the latest version. The root `README.md` ("Updating to a new Zenodo data release") uses the same pattern for the source RDF data.

In [ ]:
import json
import zipfile
from pathlib import Path

import requests

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

FULL_DIR = ROOT / "data"
CONCEPT_RECORD_ID = "20847651"  # https://doi.org/10.5281/zenodo.20847651

headers = {"User-Agent": "plantmetbench-notebook/0.1"}
rec = requests.get(f"https://zenodo.org/api/records/{CONCEPT_RECORD_ID}", headers=headers, timeout=60)
rec.raise_for_status()
rec_json = rec.json()

print(f"Resolved concept DOI 10.5281/zenodo.{CONCEPT_RECORD_ID} -> "
      f"version DOI {rec_json['doi']} (record {rec_json['id']})")
print(f"Title: {rec_json['metadata']['title']}")
print(f"Published: {rec_json['metadata'].get('publication_date')}")
print("\nFiles in this record:")
for f in rec_json["files"]:
    print(f"  {f['key']}  ({f['size'] / 1e6:.1f} MB)  md5:{f['checksum']}")


In [ ]:
# Idempotent: skips the download and unzip if heterodata.pt is already present.
if (FULL_DIR / "heterodata.pt").exists():
    print(f"{FULL_DIR / 'heterodata.pt'} already exists -- skipping download.")
else:
    zip_info = next(f for f in rec_json["files"] if f["key"].endswith(".zip"))
    zip_path = ROOT / "data" / zip_info["key"]
    zip_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"Downloading {zip_info['key']} ({zip_info['size'] / 1e6:.1f} MB)...")
    with requests.get(zip_info["links"]["self"], headers=headers, stream=True, timeout=300) as r:
        r.raise_for_status()
        with zip_path.open("wb") as fh:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                fh.write(chunk)

    print(f"Unzipping -> {FULL_DIR}")
    FULL_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(FULL_DIR)
    print("Done:")
    for p in sorted(FULL_DIR.iterdir()):
        print(f"  {p.name}")


## Part 2 -- Embedding exploration

### 2.1 Load the graph and build per-node organism labels

Each `Metabolite`, `Protein` and `GeneProduct` node has a direct `organism` edge. A few nodes have more than one (conserved across species); for plotting the first is taken, which is fine for a qualitative diagnostic but not for quantitative use. Common names are filled in for the most frequent taxa. The long tail of rarer species is grouped into `"other"` so the legend stays readable, and the `NCBITaxon_33090` fallback ("Viridiplantae", species unspecified) gets its own bucket.

In [ ]:
import numpy as np
import pandas as pd
import torch

data = torch.load(FULL_DIR / "heterodata.pt", weights_only=False)
nodes = pd.read_csv(FULL_DIR / "nodes.tsv", sep="\t")

org_nodes = (
    nodes[nodes["node_type"] == "Organism"]
    .reset_index(drop=True)
    .rename_axis("organism_idx")
    .reset_index()
)
org_nodes["taxon"] = org_nodes["node_id"].str.rsplit("/", n=1).str[-1]

FALLBACK_TAXON = "NCBITaxon_33090"  # "unknown organism" (Viridiplantae, species unspecified)

TAXON_LABELS = {
    "NCBITaxon_3702":  "A. thaliana",
    "NCBITaxon_3847":  "G. max (soybean)",
    "NCBITaxon_4577":  "Z. mays (maize)",
    "NCBITaxon_4530":  "O. sativa (rice)",
    "NCBITaxon_3888":  "P. sativum (pea)",
    "NCBITaxon_3055":  "C. reinhardtii",
    "NCBITaxon_4113":  "S. tuberosum (potato)",
    "NCBITaxon_3694":  "P. trichocarpa (poplar)",
    "NCBITaxon_4097":  "N. tabacum (tobacco)",
    "NCBITaxon_4081":  "S. lycopersicum (tomato)",
    "NCBITaxon_4565":  "T. aestivum (wheat)",
    "NCBITaxon_29760": "V. vinifera (grapevine)",
    "NCBITaxon_4513":  "H. vulgare (barley)",
    "NCBITaxon_3562":  "S. oleracea (spinach)",
    "NCBITaxon_4058":  "C. roseus",
    "NCBITaxon_4102":  "P. somniferum (poppy)",
    FALLBACK_TAXON:    "unknown (Viridiplantae)",
}
org_idx_to_label = {
    row.organism_idx: TAXON_LABELS.get(row.taxon, "other (single species)")
    for row in org_nodes.itertuples()
}


def node_organism_labels(ntype: str) -> np.ndarray:
    """First organism label per node, '(no organism edge)' if none."""
    ei = data[(ntype, "organism", "Organism")].edge_index
    first_org = {}
    for prot_idx, org_idx in zip(ei[0].tolist(), ei[1].tolist()):
        first_org.setdefault(prot_idx, org_idx)
    n = data[ntype].num_nodes
    return np.array([
        org_idx_to_label.get(first_org.get(i), "(no organism edge)") for i in range(n)
    ])


for ntype in ["Protein", "GeneProduct", "Metabolite"]:
    labels = node_organism_labels(ntype)
    vc = pd.Series(labels).value_counts()
    print(f"{ntype}: top organism labels -> {dict(vc.head(5))}")


### 2.2 PCA / t-SNE / UMAP, coloured by organism

Only **embedded** nodes are projected (non-zero feature vector) — unresolved
nodes are an exact all-zero vector by construction (see
`03_embeddings_fulldata.ipynb` §3), which would otherwise pile up
as one giant fake "cluster at the origin" and distort every projection.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from umap import UMAP

RANDOM_SEED = 42
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def embedded_mask(ntype: str) -> np.ndarray:
    x = data[ntype].x
    return (x.abs().sum(1) > 0).numpy()


def plot_projections(X: np.ndarray, labels: np.ndarray, title: str, save_name: str, max_categories: int = 14):
    """3-panel PCA / t-SNE / UMAP scatter, coloured by `labels`."""
    # Keep the legend readable: the most frequent categories get their own colour,
    # everything past max_categories is folded into "other".
    top_cats = list(pd.Series(labels).value_counts().head(max_categories).index)
    plot_labels = np.where(np.isin(labels, top_cats), labels, "other")
    cats = sorted(set(plot_labels))
    cmap = plt.get_cmap("tab20")
    colors = {c: cmap(i % 20) for i, c in enumerate(cats)}

    pca = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X)
    tsne = TSNE(n_components=2, random_state=RANDOM_SEED, init="pca", perplexity=30).fit_transform(X)
    umap_emb = UMAP(n_components=2, random_state=RANDOM_SEED).fit_transform(X)

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for ax, proj, name in zip(axes, [pca, tsne, umap_emb], ["PCA", "t-SNE", "UMAP"]):
        for c in cats:
            m = plot_labels == c
            ax.scatter(proj[m, 0], proj[m, 1], s=6, alpha=0.6, label=c, color=colors[c])
        ax.set_title(f"{name} -- {title}")
        ax.set_xticks([]); ax.set_yticks([])
    axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, markerscale=2)
    plt.tight_layout()
    fig.savefig(FIG_DIR / save_name, dpi=150, bbox_inches="tight")
    plt.show()
    return pca, tsne, umap_emb


proj_results = {}
for ntype in ["Protein", "GeneProduct", "Metabolite"]:
    mask = embedded_mask(ntype)
    X = data[ntype].x[mask].numpy()
    labels = node_organism_labels(ntype)[mask]
    print(f"\n{ntype}: projecting {mask.sum():,}/{len(mask):,} embedded nodes ({X.shape[1]}-dim)")
    proj_results[ntype] = plot_projections(
        X, labels, title=f"{ntype} embeddings by organism", save_name=f"embedding_organism_{ntype.lower()}.png"
    )


### 2.3 Interpretation

**Protein (ESM-C, sequence-based).** *A. thaliana* dominates by count, as the best-annotated reference genome (`03_embeddings_fulldata.ipynb` §2), and forms the large central mass in all three projections. UMAP pulls out a few small, distinct satellite clusters of non-Arabidopsis points. This fits the fact that ESM embeddings mainly encode sequence homology and function, with species a real but secondary axis: enzymes performing the same reaction are often conserved across species, so strong organism clustering is not expected even with perfect data.

**GeneProduct (PlantCaduceus, DNA-sequence-based).** The projections show a more clearly separated cluster than Protein does, off to one side. This is consistent with the biology: DNA-level features (codon usage, GC content, nucleotide composition) carry a stronger species signal than protein sequence, which is under stronger functional constraint across species.

**Metabolite (MAP4, structure-based).** There is essentially no organism signal, for two reasons. Organism tagging for `Metabolite` nodes is almost entirely the generic `NCBITaxon_33090` ("Viridiplantae") fallback, so there is barely any real label to colour by. And even with perfect labels, a metabolite's structure does not depend on which organism's pathway page it is annotated under (glucose is glucose). The UMAP sub-islands reflect chemical class (for example lipids, sugars and amino acids), not species.

**Consequence for the splits.** This is why the taxa-held-out split (Part 3) is the informative generalisation test for `Protein` and `GeneProduct`. If held-out organisms were indistinguishable from training organisms in embedding space, the task would be close to recall; the partial separation seen here indicates that the split asks for cross-species generalisation.

## Part 3 -- Does the taxa-held-out split carve out a distinct region?

This part reuses the Protein PCA, t-SNE and UMAP coordinates from §2.2 (`proj_results["Protein"]`) and recolours them by split membership instead of organism, so any visual difference from the §2.2 Protein panel comes from the colouring alone.

Only `splits_taxa.pt` is used. It is the one split in which each `Protein` node belongs to exactly one split (one organism maps to one split; see `03_embeddings_fulldata.ipynb` §6). `splits.pt` (random) and `splits_pathway.pt` assign individual edges to splits, so the same protein can appear in train and val/test edges at once and "this protein's split" is not a single label.

In [ ]:
taxa = torch.load(FULL_DIR / "splits_taxa.pt", weights_only=False)

protein_split = np.full(data["Protein"].num_nodes, "(not a target-edge protein)", dtype=object)
for split_name in ["train", "val", "test"]:
    idx = taxa[f"{split_name}_edge_index"][0].unique().numpy()
    conflict = (protein_split[idx] != "(not a target-edge protein)") & (protein_split[idx] != split_name)
    if conflict.any():
        print(f"WARNING: {conflict.sum()} protein(s) already assigned before '{split_name}' -- unexpected overlap")
    protein_split[idx] = split_name

print("Protein split-label counts (all nodes):")
print(pd.Series(protein_split).value_counts())

mask = embedded_mask("Protein")
split_labels_embedded = protein_split[mask]
print(f"\nOf the {mask.sum():,} embedded Protein nodes:")
print(pd.Series(split_labels_embedded).value_counts())

pca, tsne, umap_emb = proj_results["Protein"]
split_colors = {"train": "#4c72b0", "val": "#dd8452", "test": "#55a868",
                "(not a target-edge protein)": "lightgray"}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, proj, name in zip(axes, [pca, tsne, umap_emb], ["PCA", "t-SNE", "UMAP"]):
    for split_name in ["(not a target-edge protein)", "train", "val", "test"]:
        m = split_labels_embedded == split_name
        ax.scatter(proj[m, 0], proj[m, 1], s=6, alpha=0.6, label=split_name, color=split_colors[split_name])
    ax.set_title(f"{name} -- Protein embeddings by taxa-split membership")
    ax.set_xticks([]); ax.set_yticks([])
axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9, markerscale=2)
plt.tight_layout()
fig.savefig(FIG_DIR / "embedding_split_protein.png", dpi=150, bbox_inches="tight")
plt.show()


### 3.1 Interpretation

Only a small share of the embedded `Protein` nodes are `val` or `test`, because the taxa split holds out whole organisms and most organisms are small. The held-out points therefore do not form a large, sharply separated blob as the organism colouring of §2.2 does.

They are still informative. The `val` and `test` points are not spread uniformly through the train-dominated mass; several fall inside the small UMAP satellite clusters that §2.2 identified as non-Arabidopsis regions. That is the expected picture for an organism-held-out split: train is dominated by *A. thaliana* (the large central mass), and the held-out organisms' proteins occupy a real, if minority, region of embedding space that train barely touches. This is consistent with a split that tests cross-species generalisation rather than near-duplicate recall. Held-out points sitting exactly on dense train regions would instead suggest that the held-out organisms are too similar to *A. thaliana* for a meaningful test.

## Part 4 -- Conversion (reaction) embeddings, coloured by EC class

### 4.1 Load conversion embeddings and EC-class labels

`Conversion` nodes (the `Interaction` subtype that is this project's link-prediction
target) don't have a pre-attached embedding in `heterodata.pt` the way
Protein/Metabolite/GeneProduct do -- they get random 64-dim features by default
(see `training/dataset.py`). `scripts/compute_conversion_embeddings.py`
instead builds a **3072-dim substrate-product reaction fingerprint** per Conversion:
`[mean(substrate MAP4), mean(product MAP4), substrate - product]`, saved to
`data/embeddings_conversion.pt`. This is the fingerprint projected here.

Organism does not apply as a colouring dimension here (§2.2): a reaction's chemistry is the same regardless of which organism's pathway mentions it, as for the Metabolite case in §2.3. The reactions are coloured by **EC (Enzyme Commission) top-level class** instead, read from the `ec_number` column of `nodes.tsv`. This is the natural type label for a reaction, analogous to organism for a protein.

In [ ]:
EC_NAMES = {
    "1": "Oxidoreductases", "2": "Transferases", "3": "Hydrolases",
    "4": "Lyases", "5": "Isomerases", "6": "Ligases", "7": "Translocases",
}

conv_nodes = nodes[nodes["interaction_subtype"] == "Conversion"].reset_index(drop=True)
conv_ids = conv_nodes["node_id"].tolist()
ec_of_node = dict(zip(conv_nodes["node_id"], conv_nodes["ec_number"]))   # full EC string, e.g. "2.1.1.280"

conv_emb: dict[str, torch.Tensor] = torch.load(FULL_DIR / "embeddings_conversion.pt", weights_only=False)
print(f"Conversion nodes: {len(conv_ids):,}  |  fingerprints in embeddings_conversion.pt: {len(conv_emb):,}")


def conv_ec_labels(ids: list[str]) -> np.ndarray:
    """Top-level EC class name per Conversion node id, '(no EC annotation)' if none."""
    labels = []
    for nid in ids:
        ec_str = ec_of_node.get(nid)
        has_ec = isinstance(ec_str, str) and ec_str != ""
        labels.append(EC_NAMES.get(ec_str.split(".")[0], "(no EC annotation)") if has_ec else "(no EC annotation)")
    return np.array(labels)


ec_labels_all = conv_ec_labels(conv_ids)
print("\nEC class breakdown (all Conversion nodes):")
print(pd.Series(ec_labels_all).value_counts())


### 4.2 PCA / t-SNE / UMAP, coloured by EC class

Same `plot_projections()` helper as §2.2. As with the other node types, only
Conversions with a **non-zero** fingerprint are projected -- `compute_conversion_embeddings.py`
falls back to an all-zero 3072-dim vector when neither substrate nor product
metabolites resolved to a MAP4 embedding, which would otherwise pile up as a
fake cluster at the origin.

In [ ]:
conv_emb_matrix = torch.stack([conv_emb.get(nid, torch.zeros(3072)) for nid in conv_ids])
conv_mask = (conv_emb_matrix.abs().sum(1) > 0).numpy()

X = conv_emb_matrix[conv_mask].numpy()
labels = ec_labels_all[conv_mask]
print(f"Conversion: projecting {conv_mask.sum():,}/{len(conv_mask):,} nodes with a non-zero fingerprint ({X.shape[1]}-dim)")

proj_results["Conversion"] = plot_projections(
    X, labels, title="Conversion embeddings by EC class", save_name="embedding_ecclass_conversion.png"
)


### 4.3 Interpretation

**No clean separation by EC class.** In none of the three projections is there a region dominated by one enzyme class, unlike the organism clustering of GeneProduct in §2.2. This is expected, because the substrate-product MAP4 fingerprint encodes the chemical identity of the metabolites in a reaction, not the catalytic mechanism (the bond formed or broken) that defines an EC class. Two Transferase reactions with different substrates can lie far apart, while a Transferase and an Oxidoreductase that share a common cofactor (ATP, NAD+) can lie close together.

**Small, tight islands.** t-SNE and UMAP show a number of small, near-monochromatic islands, for example a dark-green Oxidoreductase cluster at the right edge of the t-SNE panel and several small red Transferase satellites. These are more likely near-duplicate reactions that share the same core substrate and product pair (the same cofactor chemistry recurring across pathway variants) than EC-class structure. The overall shape, a large central mass with small tight satellites, is the same as for Protein in §2.2, but here the satellites do not line up with the colouring.

**Class frequencies.** Oxidoreductases (green, 5,418) and Transferases (red, 4,893) account for most of the 13,585 EC-annotated Conversions, as expected for plant metabolism, and they also extend furthest along the PCA "fan" edges. They are both the most common and the most chemically diverse classes, so they appear throughout the projection instead of forming a class-specific region.

**Reactions without EC number.** The 6,342 of 19,927 Conversion nodes without an EC number (blue) are scattered throughout the projection and are not chemically distinct from the annotated reactions. The missing EC number is therefore a gap in the source annotation, not a marker of a different reaction type.

**Takeaway.** The fingerprint and the EC class are only loosely coupled, so the fingerprint alone does not reveal the reaction class. This is why EC features are kept out of the default baseline: appending an EC one-hot (`load_ec_embeddings` in `training/dataset.py`, `--ec-features` in `train.py`) gives the model the type of chemistry directly, on top of the molecules involved, and is used only as an ablation.